# ProZorro Ukraine Public Procurement - Quick Start Guide

This notebook demonstrates basic usage of the ProZorro public procurement dataset covering 2022-2025.

**Dataset highlights:**
- **13.1M+ completed tenders** from Ukrainian government procurement
- **Period**: Jan 1, 2022 - Dec 31, 2025 (includes pre-war baseline)
- **Coverage**: 37K buyers, 359K suppliers, 72K bidders
- Normalized relational structure with rich metadata
- Year-based file structure for memory efficiency

**File Structure:**
```
/kaggle/input/prozorro-ukraine-procurement-2022-2025/
├── tenders_2022.csv - tenders_2025.csv  (55 columns each)
├── bids_2022.csv - bids_2025.csv        (8 columns: incl. bid_amount, is_winner)
├── buyers.csv                            (16 columns)
├── suppliers.csv                         (6 columns)
└── bidders.csv                           (7 columns, EDRPOU-based)
```

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = '/kaggle/input/datasets/romankachmar/prozorro-ukraine-procurement-2022-2025'

## 1. Load Data

**Year-based files** - each year in separate file for memory efficiency:
- Load one year at a time (~2-3 GB RAM per year)
- Or combine multiple years with `pd.concat()`

In [ ]:
# Load single year
tenders_2023 = pd.read_csv(f'{DATA_DIR}/tenders_2023.csv', low_memory=False)
print(f"Loaded {len(tenders_2023):,} tenders from 2023")
print(f"Columns: {len(tenders_2023.columns)}")
tenders_2023.head()

In [ ]:
# Load all years combined (requires ~8 GB RAM)
# years = [2022, 2023, 2024, 2025]
# tenders_all = pd.concat([
#     pd.read_csv(f'{DATA_DIR}/tenders_{year}.csv', low_memory=False)
#     for year in years
# ], ignore_index=True)
# print(f"Loaded {len(tenders_all):,} tenders (2022-2025)")

In [ ]:
suppliers = pd.read_csv(f'{DATA_DIR}/suppliers.csv')
buyers = pd.read_csv(f'{DATA_DIR}/buyers.csv')
bidders = pd.read_csv(f'{DATA_DIR}/bidders.csv')

print(f"Suppliers: {len(suppliers):,}")
print(f"Buyers: {len(buyers):,}")
print(f"Bidders: {len(bidders):,} (unique companies by EDRPOU)")

In [ ]:
bids_2023 = pd.read_csv(f'{DATA_DIR}/bids_2023.csv')

print(f"Bids 2023: {len(bids_2023):,}")
print(f"Columns: {list(bids_2023.columns)}")
print(f"\nbid_amount coverage: {bids_2023['bid_amount'].notna().mean()*100:.1f}%")
print(f"Winners: {bids_2023['is_winner'].sum():,} ({bids_2023['is_winner'].mean()*100:.1f}%)")
print("\nSample bids:")
bids_2023[['tender_id', 'bidder_id', 'bid_amount', 'is_winner', 'bid_status']].head()

## 2. Basic Data Exploration

In [ ]:
print("Tenders dataset structure:")
print(f"Rows: {len(tenders_2023):,}")
print(f"Columns: {len(tenders_2023.columns)}")
print("\nColumn names:")
print(tenders_2023.columns.tolist())

In [ ]:
# Basic statistics
print("=" * 60)
print("BASIC STATISTICS (2023)")
print("=" * 60)

print(f"\nTotal tenders: {len(tenders_2023):,}")
print(f"Unique buyers: {tenders_2023['buyer_id'].nunique():,}")
print(f"Unique suppliers: {tenders_2023['supplier_id'].nunique():,}")

print(f"\nTender Value (UAH):")
print(f"  Mean: {tenders_2023['tender_value'].mean():,.0f}")
print(f"  Median: {tenders_2023['tender_value'].median():,.0f}")
print(f"  Total: {tenders_2023['tender_value'].sum():,.0f}")

print(f"\nAward Value (UAH):")
print(f"  Mean: {tenders_2023['award_value'].mean():,.0f}")
print(f"  Median: {tenders_2023['award_value'].median():,.0f}")
print(f"  Total: {tenders_2023['award_value'].sum():,.0f}")

print(f"\nCompetition:")
print(f"  Avg tenderers: {tenders_2023['number_of_tenderers'].mean():.2f}")
print(f"  Competitive rate: {tenders_2023['is_competitive'].mean()*100:.1f}%")
print(f"  Single bidder rate: {(tenders_2023['number_of_tenderers'] == 1).mean()*100:.1f}%")

## 3. Pre-War vs War Period Comparison

**Pre-war**: Jan 1 - Feb 23, 2022  
**War**: Feb 24, 2022 onwards

In [ ]:
# Load 2022 data for war impact analysis
tenders_2022 = pd.read_csv(f'{DATA_DIR}/tenders_2022.csv', low_memory=False)

# Convert published_date to datetime (UTC)
tenders_2022['published_date'] = pd.to_datetime(tenders_2022['published_date'], utc=True, errors='coerce')

# Define war start date (timezone-aware)
WAR_START = pd.Timestamp('2022-02-24', tz='UTC')

# Split data
pre_war = tenders_2022[tenders_2022['published_date'] < WAR_START]
war = tenders_2022[tenders_2022['published_date'] >= WAR_START]

print(f"Pre-war tenders (Jan 1 - Feb 23, 2022): {len(pre_war):,}")
print(f"War period tenders (Feb 24 - Dec 31, 2022): {len(war):,}")
print(f"\nChange: {(len(war) - len(pre_war)) / len(pre_war) * 100:.1f}%")

In [ ]:
# Compare key metrics
comparison = pd.DataFrame({
    'Pre-War': [
        len(pre_war),
        pre_war['tender_value'].mean(),
        pre_war['number_of_tenderers'].mean(),
        pre_war['is_competitive'].mean() * 100
    ],
    'War Period': [
        len(war),
        war['tender_value'].mean(),
        war['number_of_tenderers'].mean(),
        war['is_competitive'].mean() * 100
    ]
}, index=['Total Tenders', 'Avg Tender Value (UAH)', 'Avg Tenderers', 'Competitive Rate (%)'])

comparison['Change (%)'] = ((comparison['War Period'] - comparison['Pre-War']) / comparison['Pre-War'] * 100)

print("\nPRE-WAR vs WAR PERIOD COMPARISON")
print("=" * 70)
print(comparison.to_string())

## 4. Top Buyers Analysis

In [ ]:
# Top 10 buyers by tender count
top_buyers = buyers.nlargest(10, 'total_tenders')[[
    'buyer_name', 'total_tenders', 'total_value', 
    'competitive_rate', 'avg_discount_pct'
]]

print("TOP 10 BUYERS (by tender count)")
print("=" * 80)
print(top_buyers.to_string(index=False))

In [ ]:
# Visualize top buyers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Truncate long names
def truncate(name, max_len=40):
    return name[:max_len] + '...' if len(str(name)) > max_len else name

# Top 10 by tender count
top_10_count = buyers.nlargest(10, 'total_tenders').copy()
top_10_count['short_name'] = top_10_count['buyer_name'].apply(truncate)
axes[0].barh(top_10_count['short_name'][::-1], top_10_count['total_tenders'][::-1])
axes[0].set_xlabel('Total Tenders')
axes[0].set_title('Top 10 Buyers by Tender Count')
axes[0].grid(axis='x', alpha=0.3)

# Top 10 by total value
top_10_value = buyers.nlargest(10, 'total_value').copy()
top_10_value['short_name'] = top_10_value['buyer_name'].apply(truncate)
axes[1].barh(top_10_value['short_name'][::-1], top_10_value['total_value'][::-1] / 1e9)
axes[1].set_xlabel('Total Value (Billion UAH)')
axes[1].set_title('Top 10 Buyers by Total Value')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Top Suppliers Analysis

In [ ]:
# Top 10 suppliers by awards
top_suppliers = suppliers.nlargest(10, 'total_awards')[[
    'supplier_name', 'supplier_region', 'total_awards', 'total_value'
]]

print("TOP 10 SUPPLIERS (by awards won)")
print("=" * 80)
print(top_suppliers.to_string(index=False))

## 6. Joining Tables

The dataset uses relational structure - you can join tables by IDs:

In [ ]:
# Join tenders with buyer details
tenders_with_buyer = tenders_2023.merge(
    buyers[['buyer_id', 'buyer_name', 'buyer_region']], 
    on='buyer_id', 
    how='left'
)

print(f"Joined tenders with buyer details: {len(tenders_with_buyer):,} rows")
print("\nSample:")
tenders_with_buyer[['tender_id', 'buyer_name', 'tender_value', 'award_value']].head()

In [ ]:
# Join with supplier details
tenders_full = tenders_with_buyer.merge(
    suppliers[['supplier_id', 'supplier_name', 'supplier_region']], 
    on='supplier_id', 
    how='left',
    suffixes=('_buyer', '_supplier')
)

print(f"Full dataset with buyer and supplier: {len(tenders_full):,} rows")
tenders_full[['tender_id', 'buyer_name', 'supplier_name', 'tender_value']].head()

## 7. Regional Analysis

In [ ]:
# Procurement by buyer region
regional_stats = tenders_2023.merge(buyers[['buyer_id', 'buyer_region']], on='buyer_id')
regional_summary = regional_stats.groupby('buyer_region').agg({
    'tender_id': 'count',
    'tender_value': 'sum',
    'number_of_tenderers': 'mean',
    'is_competitive': 'mean'
}).round(2)

regional_summary.columns = ['Total Tenders', 'Total Value (UAH)', 'Avg Tenderers', 'Competitive Rate']
regional_summary['Competitive Rate'] = regional_summary['Competitive Rate'] * 100
regional_summary = regional_summary.sort_values('Total Tenders', ascending=False)

print("\nPROCUREMENT BY REGION (2023)")
print("=" * 80)
print(regional_summary.head(15).to_string())

## 8. Fraud Indicators

Several columns can indicate potential anomalies:

In [ ]:
# Red flags analysis
print("POTENTIAL FRAUD INDICATORS (2023)")
print("=" * 60)

# 1. Single bidder tenders
single_bidder = (tenders_2023['number_of_tenderers'] == 1).sum()
print(f"\n1. Single bidder tenders: {single_bidder:,} ({single_bidder/len(tenders_2023)*100:.1f}%)")

# 2. Zero tenderers (direct award)
zero_tenderers = (tenders_2023['number_of_tenderers'] == 0).sum()
print(f"2. Zero tenderers (direct): {zero_tenderers:,} ({zero_tenderers/len(tenders_2023)*100:.1f}%)")

# 3. Award > Tender value
award_exceeds = (tenders_2023['award_value'] > tenders_2023['tender_value']).sum()
print(f"3. Award > Tender value: {award_exceeds:,} ({award_exceeds/len(tenders_2023)*100:.2f}%)")

# 4. High discount (potential underbidding)
high_discount = (tenders_2023['price_change_pct'] > 30).sum()
print(f"4. High discount (>30%): {high_discount:,} ({high_discount/len(tenders_2023)*100:.2f}%)")

# 5. Negative discount (price increase)
negative_discount = (tenders_2023['price_change_pct'] < 0).sum()
print(f"5. Negative discount: {negative_discount:,} ({negative_discount/len(tenders_2023)*100:.2f}%)")

# 6. Non-competitive tenders
non_competitive = (tenders_2023['is_competitive'] == 0).sum()
print(f"6. Non-competitive: {non_competitive:,} ({non_competitive/len(tenders_2023)*100:.1f}%)")

## 9. Temporal Trends

In [ ]:
# Monthly tender volume in 2023
tenders_2023['published_date'] = pd.to_datetime(tenders_2023['published_date'], utc=True, errors='coerce')
tenders_2023['month_period'] = tenders_2023['published_date'].dt.to_period('M')

monthly_stats = tenders_2023.groupby('month_period').agg({
    'tender_id': 'count',
    'tender_value': 'sum'
})

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Tender count
monthly_stats['tender_id'].plot(ax=axes[0], marker='o')
axes[0].set_ylabel('Number of Tenders')
axes[0].set_title('Monthly Tender Volume (2023)')
axes[0].grid(alpha=0.3)

# Total value
(monthly_stats['tender_value'] / 1e9).plot(ax=axes[1], marker='o', color='green')
axes[1].set_ylabel('Total Value (Billion UAH)')
axes[1].set_xlabel('Month')
axes[1].set_title('Monthly Procurement Value (2023)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()